# Dhaka House Rent Prediction

**Goal:** Predict the monthly rent (BDT) of an apartment in Dhaka from its location, size, bedrooms and bathrooms.

**Domain:** Real estate (Bangladesh)  
**Type:** Supervised learning, regression  
**Target:** `Price` (monthly rent in BDT)  
**Features:** `Location` (grouped into main area), `Area` (sqft), `Bed`, `Bath`  
**Data:** 28,800 rental listings (Dhaka)  
**Selection rule:** among models whose train vs test gap is below 5%, choose the one with the lowest error (MAE).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load the data
In Google Colab, a file picker will appear. Select `DATASET_houserentdhaka.csv`.

In [ ]:
import os
CSV = 'DATASET_houserentdhaka.csv'

try:
    from google.colab import files
    if not os.path.exists(CSV):
        files.upload()          # choose DATASET_houserentdhaka.csv
except ImportError:
    pass                         # running locally, file is in the same folder

df = pd.read_csv(CSV)
print(df.shape)
df.head()

## 2. Understand the data

In [ ]:
df.info()
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows (excluding index column):',
      df.drop(columns=['Unnamed: 0']).duplicated().sum())

**Observation:** There are no missing values, but nearly half of the rows are exact duplicates. 
Duplicates must be removed *before* the train/test split, otherwise the same listing can appear in both sets and the test score looks better than it really is.

## 3. Clean the data

In [ ]:
# 1) Drop the leftover index column
df = df.drop(columns=['Unnamed: 0'])

# 2) Convert Price text ("20 Thousand", "1.5 Lakh") to numbers
def price_to_num(s):
    n = float(s.split()[0].replace(',', ''))
    return n * 1000 if 'Thousand' in s else n * 100000

df['Price'] = df['Price'].apply(price_to_num)

# 3) Convert Area text ("1,600 sqft") to a number
df['Area'] = df['Area'].str.replace(' sqft', '').str.replace(',', '').astype(float)

# 4) Group 730 detailed locations into the main area (e.g. Mirpur, Gulshan)
parts = df['Location'].str.split(',')
df['Area_Name'] = parts.apply(lambda p: p[-2].strip() if len(p) >= 2 else p[0].strip())

# 5) Remove duplicates
before = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', before - len(df))

# 6) Remove extreme outliers (rent outside 1st-99th percentile, unrealistic size)
low, high = df['Price'].quantile([0.01, 0.99])
df = df[(df['Price'] >= low) & (df['Price'] <= high)]
df = df[(df['Area'] >= 200) & (df['Area'] <= 4000)]

print('Final shape:', df.shape)
print('Main areas:', df['Area_Name'].nunique())
df.head()

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

sns.histplot(df['Price'], bins=40, ax=ax[0], color='#2a6f97')
ax[0].set_title('Rent distribution (BDT)')

top = df.groupby('Area_Name')['Price'].agg(['mean', 'count'])
top = top[top['count'] >= 50].sort_values('mean', ascending=False).head(10)
sns.barplot(x=top['mean'], y=top.index, ax=ax[1], color='#2a6f97')
ax[1].set_title('Top 10 areas by average rent (50+ listings)')
ax[1].set_xlabel('Average rent (BDT)')

sns.scatterplot(data=df.sample(3000, random_state=1), x='Area', y='Price', alpha=.4, ax=ax[2])
ax[2].set_title('Size vs rent')

plt.tight_layout()
plt.show()

In [ ]:
print(df[['Area', 'Bed', 'Bath', 'Price']].corr()['Price'].round(2))

**Observation:** Rent is right-skewed (a few very expensive flats), so the model is trained on `log(Price)`. 
Location and size have the strongest effect on rent. Premium areas such as Gulshan and Banani are much more expensive than Mirpur or Uttara.

## 5. Prepare features

In [ ]:
# One-hot encode the main area (no fake numeric order like LabelEncoder)
X = pd.get_dummies(df[['Area_Name', 'Area', 'Bed', 'Bath']], columns=['Area_Name'])
y = np.log1p(df['Price'])            # log target handles the skewed rent

feature_cols = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 6. Train and compare models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest (default)': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest (tuned)': RandomForestRegressor(n_estimators=200, max_depth=12,
                                                   min_samples_leaf=10, random_state=42, n_jobs=-1),
    'XGBoost (tuned)': XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
                                    reg_lambda=5, random_state=42, n_jobs=-1),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    tr_r2 = r2_score(y_train, m.predict(X_train))
    te_r2 = r2_score(y_test, m.predict(X_test))
    # Errors in real Taka (convert back from log)
    actual = np.expm1(y_test)
    pred = np.expm1(m.predict(X_test))
    rows.append({
        'Model': name,
        'Train R2': round(tr_r2, 3),
        'Test R2': round(te_r2, 3),
        'Gap (%)': round((tr_r2 - te_r2) * 100, 1),
        'MAE (BDT)': round(mean_absolute_error(actual, pred)),
        'RMSE (BDT)': round(np.sqrt(mean_squared_error(actual, pred))),
    })

results = pd.DataFrame(rows)
results

**How to read this table**
- **R2:** closer to 1 is better (computed on log rent).
- **Gap:** Train R2 minus Test R2. A big gap means overfitting. Our limit is 5%.
- **MAE:** average error in Taka.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
results.set_index('Model')[['Train R2', 'Test R2']].plot(kind='bar', ax=ax[0], rot=20)
ax[0].set_title('Train vs Test R2')
ax[0].set_ylim(0.6, 1)
results.set_index('Model')['MAE (BDT)'].plot(kind='bar', ax=ax[1], rot=20, color='#d97706')
ax[1].set_title('Mean absolute error (BDT)')
plt.tight_layout()
plt.show()

## 7. Choose the best model

In [ ]:
eligible = results[results['Gap (%)'] < 5]
best_row = eligible.sort_values('MAE (BDT)').iloc[0]
best_name = best_row['Model']
best_model = models[best_name]

print('Best model:', best_name)
print(best_row.to_string())

**Decision:** Random Forest (default) overfits (large train vs test gap), so it is rejected. 
Among the models whose gap is below 5%, the one with the lowest average error (MAE) is selected. Linear Regression and XGBoost have the same test R2, but XGBoost makes smaller errors in Taka and can also capture non-linear patterns, so it is preferred.

In [ ]:
# Actual vs predicted on the test set (in BDT)
actual = np.expm1(y_test)
pred = np.expm1(best_model.predict(X_test))

plt.figure(figsize=(5.5, 5.5))
plt.scatter(actual, pred, alpha=.3, s=10)
lim = [0, max(actual.max(), pred.max())]
plt.plot(lim, lim, 'r--')
plt.xlabel('Actual rent (BDT)'); plt.ylabel('Predicted rent (BDT)')
plt.title(f'{best_name}: actual vs predicted')
plt.show()

In [ ]:
# Which features matter most? (works for tree models)
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
    imp.sort_values().plot(kind='barh', figsize=(6, 4), color='#2a6f97')
    plt.title('Top 10 important features')
    plt.show()

## 8. Save the model

In [ ]:
areas = sorted(df['Area_Name'].unique())

joblib.dump({'model': best_model, 'columns': feature_cols, 'areas': areas}, 'model.pkl')
print('Saved model.pkl')

## 9. Test a prediction

In [ ]:
def predict_rent(area_name, area_sqft, bed, bath):
    row = pd.DataFrame(0, index=[0], columns=feature_cols)
    row['Area'] = area_sqft
    row['Bed'] = bed
    row['Bath'] = bath
    col = f'Area_Name_{area_name}'
    if col in row.columns:
        row[col] = 1
    return float(np.expm1(best_model.predict(row)[0]))

for a, s, b, ba in [('Mirpur', 900, 2, 2), ('Gulshan', 1800, 3, 3), ('Uttara', 1200, 3, 2)]:
    print(f'{a:10s} {s} sqft, {b} bed, {ba} bath -> about {predict_rent(a, s, b, ba):,.0f} BDT / month')

## 10. Deployment (Flask web app)

The next cells create `app.py`, `templates/index.html` and `requirements.txt`. 
Run the app locally (not inside Colab):

```
pip install -r requirements.txt
python app.py
```
Then open `http://127.0.0.1:5000`.

In [ ]:
%%writefile app.py
from flask import Flask, render_template, request
import joblib, numpy as np, pandas as pd

app = Flask(__name__)

bundle = joblib.load('model.pkl')
model, columns, areas = bundle['model'], bundle['columns'], bundle['areas']


def predict_rent(area_name, area_sqft, bed, bath):
    row = pd.DataFrame(0, index=[0], columns=columns)
    row['Area'] = area_sqft
    row['Bed'] = bed
    row['Bath'] = bath
    col = f'Area_Name_{area_name}'
    if col in row.columns:
        row[col] = 1
    return float(np.expm1(model.predict(row)[0]))


@app.route('/', methods=['GET', 'POST'])
def home():
    result, error, form = None, None, {}
    if request.method == 'POST':
        form = request.form
        try:
            area_name = form['area_name']
            size = float(form['size'].replace(',', ''))
            bed = int(form['bed'])
            bath = int(form['bath'])
        except (KeyError, ValueError):
            error = 'Please enter valid numbers.'
        else:
            if area_name not in areas:
                error = 'Please select a valid area.'
            elif not 200 <= size <= 4000:
                error = 'Size must be between 200 and 4000 sqft.'
            elif not (1 <= bed <= 6 and 1 <= bath <= 7):
                error = 'Bedrooms must be 1-6 and bathrooms 1-7.'
            else:
                result = int(round(predict_rent(area_name, size, bed, bath) / 100) * 100)
    return render_template('index.html', areas=areas, result=result, error=error, form=form)


if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
import os
os.makedirs('templates', exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Dhaka House Rent Predictor</title>
  <style>
    body { font-family: Arial, sans-serif; background: #f3f6f9; margin: 0; padding: 24px; }
    .card { max-width: 420px; margin: 30px auto; background: #fff; padding: 24px;
            border-radius: 12px; box-shadow: 0 2px 10px rgba(0,0,0,.08); }
    h1 { font-size: 22px; margin-top: 0; color: #1d3557; }
    label { display: block; margin: 14px 0 4px; font-size: 14px; color: #333; }
    select, input { width: 100%; padding: 10px; border: 1px solid #ccd; border-radius: 8px;
                    font-size: 15px; box-sizing: border-box; }
    button { margin-top: 20px; width: 100%; padding: 12px; background: #2a6f97; color: #fff;
             border: 0; border-radius: 8px; font-size: 16px; cursor: pointer; }
    .result { margin-top: 20px; padding: 14px; background: #e8f4ea; border-radius: 8px;
              font-size: 18px; text-align: center; color: #1b5e20; }
    .error { margin-top: 20px; padding: 12px; background: #fdecea; border-radius: 8px; color: #b71c1c; }
  </style>
</head>
<body>
  <div class="card">
    <h1>Dhaka House Rent Predictor</h1>
    <form method="POST">
      <label>Area</label>
      <select name="area_name" required>
        {% for a in areas %}
        <option value="{{ a }}" {% if form.get('area_name') == a %}selected{% endif %}>{{ a }}</option>
        {% endfor %}
      </select>

      <label>Size (sqft)</label>
      <input type="text" name="size" value="{{ form.get('size', '1000') }}" required>

      <label>Bedrooms</label>
      <input type="number" name="bed" min="1" max="6" value="{{ form.get('bed', '2') }}" required>

      <label>Bathrooms</label>
      <input type="number" name="bath" min="1" max="7" value="{{ form.get('bath', '2') }}" required>

      <button type="submit">Predict Rent</button>
    </form>

    {% if result %}
      <div class="result">Estimated rent: <b>{{ "{:,}".format(result) }} BDT / month</b></div>
    {% endif %}
    {% if error %}
      <div class="error">{{ error }}</div>
    {% endif %}
  </div>
</body>
</html>

In [ ]:
%%writefile requirements.txt
flask
pandas
numpy
scikit-learn
xgboost
joblib

## 11. Conclusion

- The raw data contained about 47% duplicate rows, which were removed before splitting.
- Grouping 730 detailed locations into main areas and using a log-transformed target gave a much better and more stable model.
- The default Random Forest overfitted; tuned models reduced the train vs test gap to under 5%.
- The final model is deployed as a Flask web app where a user enters area, size, bedrooms and bathrooms to get an estimated monthly rent.

**Limitations:** the data only contains rent listings (not sale prices) and has no floor, age or amenities information, so predictions are estimates (typical error is shown by MAE above).